# 01 — Dataset Familiarization
**Cities:** Singapore · Bangkok  
**Purpose:** Document schema, null rates, row counts, PK/FK relationships, and structural differences before any pipeline work.

**Files covered:**
| File | Singapore rows | Bangkok rows |
|---|---|---|
| listings.csv | 3,710 | 28,978 |
| reviews.csv | 38,350 | 583,333 |
| neighbourhoods.csv | 55 | 50 |
| calendar.csv.gz | TBD | TBD |
| neighbourhoods.geojson | — | — |

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

RAW = Path('../data/raw')
SG  = RAW / 'singapore'
BK  = RAW / 'bangkok'

---
## 1. Load All Files

In [ ]:
sg_listings       = pd.read_csv(SG / 'listings.csv', low_memory=False)
sg_reviews        = pd.read_csv(SG / 'reviews.csv')
sg_neighbourhoods = pd.read_csv(SG / 'neighbourhoods.csv')
sg_calendar       = pd.read_csv(SG / 'calendar.csv.gz', compression='gzip', low_memory=False)

bk_listings       = pd.read_csv(BK / 'listings.csv', low_memory=False)
bk_reviews        = pd.read_csv(BK / 'reviews.csv')
bk_neighbourhoods = pd.read_csv(BK / 'neighbourhoods.csv')
bk_calendar       = pd.read_csv(BK / 'calendar.csv.gz', compression='gzip', low_memory=False)

print('All files loaded.')

---
## 2. Row Counts & Shape

In [ ]:
summary = pd.DataFrame({
    'File': ['listings', 'reviews', 'neighbourhoods', 'calendar'],
    'Singapore rows': [
        len(sg_listings), len(sg_reviews), len(sg_neighbourhoods), len(sg_calendar)
    ],
    'Bangkok rows': [
        len(bk_listings), len(bk_reviews), len(bk_neighbourhoods), len(bk_calendar)
    ]
})
summary['SG cols'] = [
    sg_listings.shape[1], sg_reviews.shape[1], sg_neighbourhoods.shape[1], sg_calendar.shape[1]
]
summary['BK cols'] = [
    bk_listings.shape[1], bk_reviews.shape[1], bk_neighbourhoods.shape[1], bk_calendar.shape[1]
]
summary

---
## 3. Listings Schema — Column Types & Sample Values

In [ ]:
def schema_profile(df, name):
    """Return dtype + sample value for every column."""
    rows = []
    for col in df.columns:
        sample = df[col].dropna().iloc[0] if df[col].notna().any() else 'ALL NULL'
        rows.append({'column': col, 'dtype': str(df[col].dtype), 'sample': sample})
    result = pd.DataFrame(rows)
    result.index.name = name
    return result

print('=== SINGAPORE listings ===')
display(schema_profile(sg_listings, 'singapore'))

print('\n=== BANGKOK listings ===')
display(schema_profile(bk_listings, 'bangkok'))

---
## 4. Null Rate Analysis — Listings

In [ ]:
def null_report(df, city):
    nulls = df.isnull().sum()
    pct   = (nulls / len(df) * 100).round(2)
    return pd.DataFrame({'null_count': nulls, 'null_pct': pct}).query('null_count > 0').sort_values('null_pct', ascending=False).assign(city=city)

sg_nulls = null_report(sg_listings, 'Singapore')
bk_nulls = null_report(bk_listings, 'Bangkok')

print('=== SINGAPORE — columns with nulls ===')
display(sg_nulls)

print('\n=== BANGKOK — columns with nulls ===')
display(bk_nulls)

---
## 5. Schema Comparison — Singapore vs Bangkok

In [ ]:
sg_cols = set(sg_listings.columns)
bk_cols = set(bk_listings.columns)

only_sg = sg_cols - bk_cols
only_bk = bk_cols - sg_cols
shared  = sg_cols & bk_cols

print(f'Shared columns ({len(shared)}):  {sorted(shared)}')
print(f'\nOnly in Singapore ({len(only_sg)}): {only_sg}')
print(f'Only in Bangkok   ({len(only_bk)}): {only_bk}')

---
## 6. PK / FK Relationship Map

In [ ]:
# listings.id  ←→  reviews.listing_id  (one-to-many)
# listings.id  ←→  calendar.listing_id (one-to-many, 365 rows per listing)
# listings.neighbourhood ←→ neighbourhoods.neighbourhood (lookup)

for city, lst, rev, cal, nb in [
    ('Singapore', sg_listings, sg_reviews, sg_calendar, sg_neighbourhoods),
    ('Bangkok',   bk_listings, bk_reviews, bk_calendar, bk_neighbourhoods)
]:
    listing_ids = set(lst['id'])
    rev_ids     = set(rev['listing_id'])
    cal_ids     = set(cal['listing_id'])
    nb_names    = set(nb['neighbourhood'])
    lst_nb      = set(lst['neighbourhood'].dropna())

    print(f'\n=== {city} ===')
    print(f'  Listings with at least one review : {len(listing_ids & rev_ids)} / {len(listing_ids)}')
    print(f'  Listings with calendar entries    : {len(listing_ids & cal_ids)} / {len(listing_ids)}')
    print(f'  Review listing_ids not in listings: {len(rev_ids - listing_ids)}')
    print(f'  Calendar IDs not in listings      : {len(cal_ids - listing_ids)}')
    print(f'  Neighbourhood match (listing→nb)  : {len(lst_nb & nb_names)} of {len(lst_nb)} distinct values matched')

---
## 7. Cardinality Check — Key Categorical Columns

In [ ]:
cat_cols = ['room_type', 'neighbourhood_group', 'neighbourhood', 'license']

for col in cat_cols:
    if col in sg_listings.columns and col in bk_listings.columns:
        sg_vc = sg_listings[col].value_counts(dropna=False)
        bk_vc = bk_listings[col].value_counts(dropna=False)
        print(f'\n--- {col} ---')
        print(f'Singapore ({len(sg_vc)} unique):'); print(sg_vc.to_string())
        print(f'Bangkok   ({len(bk_vc)} unique):'); print(bk_vc.to_string())

---
## 8. Price Column Inspection

In [ ]:
for city, df in [('Singapore', sg_listings), ('Bangkok', bk_listings)]:
    print(f'\n=== {city} — price column ===')
    print(f'  dtype        : {df["price"].dtype}')
    print(f'  sample values: {df["price"].dropna().head(10).tolist()}')
    print(f'  null count   : {df["price"].isnull().sum()}')
    print(f'  zero count   : {(df["price"] == 0).sum()}')

---
## 9. Coordinate Validation

In [ ]:
# Singapore expected: lat ~1.3N, lng ~103.8E
# Bangkok expected:   lat ~13.7N, lng ~100.5E

sg_lat_ok = sg_listings['latitude'].between(1.1, 1.5).all()
sg_lng_ok = sg_listings['longitude'].between(103.5, 104.1).all()
bk_lat_ok = bk_listings['latitude'].between(13.3, 14.1).all()
bk_lng_ok = bk_listings['longitude'].between(100.1, 100.9).all()

print(f'Singapore lat OK: {sg_lat_ok} | lng OK: {sg_lng_ok}')
print(f'Bangkok   lat OK: {bk_lat_ok} | lng OK: {bk_lng_ok}')

# Flag any outlier coordinates
sg_bad = sg_listings[~sg_listings['latitude'].between(1.1, 1.5) | ~sg_listings['longitude'].between(103.5, 104.1)]
bk_bad = bk_listings[~bk_listings['latitude'].between(13.3, 14.1) | ~bk_listings['longitude'].between(100.1, 100.9)]
print(f'\nSingapore bad coordinates: {len(sg_bad)}')
print(f'Bangkok   bad coordinates: {len(bk_bad)}')

---
## 10. Calendar File Overview

In [ ]:
for city, cal in [('Singapore', sg_calendar), ('Bangkok', bk_calendar)]:
    print(f'\n=== {city} calendar ===')
    print(f'  Shape  : {cal.shape}')
    print(f'  Columns: {cal.columns.tolist()}')
    display(cal.head(3))
    print(f'  Null rates:')
    print((cal.isnull().mean() * 100).round(2))

---
## 11. Assumptions & Decisions Log

| # | Field | Assumption | Rationale |
|---|---|---|---|
| 1 | `price` = null | Drop rows where price is null — cannot be imputed meaningfully | Price is the core metric; a null price listing is unusable for analysis |
| 2 | `price` = 0 | Flag as suspicious, treat separately | $0 price likely a data entry error or blocked listing |
| 3 | `reviews_per_month` = null | Fill with 0 | Null implies no reviews yet, not missing data |
| 4 | `last_review` = null | Leave null; do not impute | A null last_review means the listing has never been reviewed |
| 5 | `availability_365` | Used as occupancy proxy: `occupancy_proxy = (365 - availability_365) / 365` | Inside Airbnb methodology; not actual bookings |
| 6 | `license` field | Treat null as unlicensed in Singapore context; field meaning differs between cities | Singapore requires STR license; Bangkok has different regulation |
| 7 | `reviews.csv` | Contains listing_id + date only (no review text) — sufficient for volume/recency analysis | Detailed review text would require reviews.csv.gz (future work) |

---
## 12. Dataset Limitations

- **Point-in-time snapshot:** Inside Airbnb scrapes data on a single date — no historical listing data
- **Reviews as booking proxy:** Review count underestimates true bookings (not all guests leave reviews)
- **No booking/revenue data:** Occupancy is estimated, not observed
- **Scraping artifacts:** Some prices may be stale if hosts haven't updated their listing
- **Bangkok scale difference:** Bangkok has ~8x more listings than Singapore — distributions and outlier thresholds will differ